In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn thop pandas -q


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Subset

import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import time
from thop import profile
import copy
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

Using device: cpu
CUDA available: False


In [ ]:


def get_data_loaders(dataset_name='MNIST', batch_size=16, pin_memory=True):

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    if dataset_name == 'MNIST':
        dataset = datasets.MNIST(root='./data', train=True,
                                 download=True, transform=transform)
        test_dataset = datasets.MNIST(root='./data', train=False,
                                      download=True, transform=transform)
    else:
        dataset = datasets.FashionMNIST(root='./data', train=True,
                                        download=True, transform=transform)
        test_dataset = datasets.FashionMNIST(root='./data', train=False,
                                             download=True, transform=transform)



    train_size = int(0.875 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, pin_memory=pin_memory, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, pin_memory=pin_memory, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, pin_memory=pin_memory, num_workers=2)

    return train_loader, val_loader, test_loader

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=10, device='cuda', use_amp=True):


    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type=='cuda')

    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    start_time = time.time()

    for epoch in range(num_epochs):

        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            if use_amp and device.type == 'cuda':
                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}] '
                  f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
                  f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    training_time = (time.time() - start_time) * 1000
    model.load_state_dict(best_model_wts)

    return model, train_losses, val_losses, train_accs, val_accs, training_time

In [ ]:
def evaluate_model(model, test_loader, device='cuda'):
    """Evaluate model on test set"""
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy

In [ ]:
def calculate_flops(model, device='cuda'):
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    flops, params = profile(model, inputs=(dummy_input,), verbose=False)
    return flops

In [ ]:
def run_q1a_all_experiments(dataset_name='MNIST', num_epochs=15):
    """
    Run ALL experiments for Q1(a) and fill the complete table
    """
    print(f"\n{'='*100}")
    print(f"Q1(a): Running ALL experiments for {dataset_name} dataset")
    print(f"{'='*100}\n")

    configs = [
        (16, 'SGD', 0.001),
        (16, 'SGD', 0.0001),
        (16, 'Adam', 0.001),
        (16, 'Adam', 0.0001),
        (32, 'SGD', 0.001),

        (32, 'Adam', 0.0001),
    ]

    results = []

    for batch_size, optimizer_name, lr in configs:
        for model_name in ['ResNet18', 'ResNet50']:
            print(f"\n{'*'*80}")
            print(f"Config: {dataset_name} | {model_name} | BS={batch_size} | "
                  f"Opt={optimizer_name} | LR={lr} | Epochs={num_epochs}")
            print(f"{'*'*80}")

            try:
                train_loader, val_loader, test_loader = get_data_loaders(
                    dataset_name, batch_size, pin_memory=True)

                if model_name == 'ResNet18':
                    model = models.resnet18(pretrained=False)
                else:
                    model = models.resnet50(pretrained=False)

                model.fc = nn.Linear(model.fc.in_features, 10)
                model = model.to(device)

                if optimizer_name == 'SGD':
                    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                else:
                    optimizer = optim.Adam(model.parameters(), lr=lr)

                criterion = nn.CrossEntropyLoss()

                model, train_losses, val_losses, train_accs, val_accs, train_time = train_model(
                    model, train_loader, val_loader, criterion, optimizer,
                    num_epochs, device, use_amp=True)

                test_acc = evaluate_model(model, test_loader, device)

                print(f"Test Accuracy: {test_acc:.2f}%")
                print(f"Training Time: {train_time:.2f} ms")

                results.append({
                    'Dataset': dataset_name,
                    'Batch_Size': batch_size,
                    'Optimizer': optimizer_name,
                    'Learning_Rate': lr,
                    'Model': model_name,
                    'Test_Accuracy': round(test_acc, 2),
                    'Train_Time_ms': round(train_time, 2)
                })

                if test_acc > 80:
                    model_filename = f'best_{dataset_name}_{model_name}_bs{batch_size}_{optimizer_name}_lr{lr}.pth'
                    torch.save(model.state_dict(), model_filename)
                    print(f"Model saved: {model_filename}")

            except Exception as e:
                print(f"✗ Error: {e}")
                results.append({
                    'Dataset': dataset_name,
                    'Batch_Size': batch_size,
                    'Optimizer': optimizer_name,
                    'Learning_Rate': lr,
                    'Model': model_name,
                    'Test_Accuracy': 0,
                    'Train_Time_ms': 0
                })

    return pd.DataFrame(results)

In [ ]:
def run_q1a_resnet50_adam_pin_false(dataset_name='MNIST', num_epochs=15): # for pin memory =false on best model

    print(f"\n{'='*100}")
    print(f"Q1(a): Running experiment for {dataset_name} dataset (ResNet50, BS=32, Adam)")
    print(f"{'='*100}\n")

    learning_rates = [0.001]
    results = []

    for lr in learning_rates:
        print(f"\n{'*'*80}")
        print(f"Config: {dataset_name} | ResNet50 | BS=32 | Opt=Adam | LR={lr} | Epochs={num_epochs}")
        print(f"{'*'*80}")

        try:
            # Load data with fixed batch size and pin_memory=False
            train_loader, val_loader, test_loader = get_data_loaders(
                dataset_name, batch_size=32, pin_memory=False
            )

            # Initialize ResNet50
            model = models.resnet50(pretrained=False)
            model.fc = nn.Linear(model.fc.in_features, 10)
            model = model.to(device)

            optimizer = optim.Adam(model.parameters(), lr=lr)
            criterion = nn.CrossEntropyLoss()

            model, train_losses, val_losses, train_accs, val_accs, train_time = train_model(
                model, train_loader, val_loader, criterion, optimizer,
                num_epochs, device, use_amp=True
            )

            test_acc = evaluate_model(model, test_loader, device)

            print(f"Test Accuracy: {test_acc:.2f}%")
            print(f"Training Time: {train_time:.2f} ms")

            results.append({
                'Dataset': dataset_name,
                'Batch_Size': 32,
                'Optimizer': 'Adam',
                'Learning_Rate': lr,
                'Model': 'ResNet50',
                'Test_Accuracy': round(test_acc, 2),
                'Train_Time_ms': round(train_time, 2)
            })

            if test_acc > 80:
                model_filename = f'best_{dataset_name}_ResNet50_bs32_Adam_lr{lr}.pth'
                torch.save(model.state_dict(), model_filename)
                print(f"Model saved: {model_filename}")

        except Exception as e:
            print(f"✗ Error: {e}")
            results.append({
                'Dataset': dataset_name,
                'Batch_Size': 32,
                'Optimizer': 'Adam',
                'Learning_Rate': lr,
                'Model': 'ResNet50',
                'Test_Accuracy': 0,
                'Train_Time_ms': 0
            })

    return pd.DataFrame(results)

In [ ]:
def run_q1b_svm_experiments():

    print(f"\n{'='*100}")
    print(f"Q1(b): Running ALL SVM experiments")
    print(f"{'='*100}\n")

    results = []

    for dataset_name in ['MNIST', 'FashionMNIST']:
        print(f"\n--- Dataset: {dataset_name} ---")

        transform = transforms.Compose([
            transforms.ToTensor(),
        ])

        if dataset_name == 'MNIST':
            train_dataset = datasets.MNIST(root='./data', train=True,
                                          download=True, transform=transform)
            test_dataset = datasets.MNIST(root='./data', train=False,
                                         download=True, transform=transform)
        else:
            train_dataset = datasets.FashionMNIST(root='./data', train=True,
                                                 download=True, transform=transform)
            test_dataset = datasets.FashionMNIST(root='./data', train=False,
                                                download=True, transform=transform)

        X_train = train_dataset.data.numpy().reshape(len(train_dataset), -1) / 255.0
        y_train = train_dataset.targets.numpy()
        X_test = test_dataset.data.numpy().reshape(len(test_dataset), -1) / 255.0
        y_test = test_dataset.targets.numpy()



        for kernel in ['poly', 'rbf']:
            for C in [0.1, 1.0, 10.0]:
                print(f"  Training SVM: kernel={kernel}, C={C}")

                try:
                    start_time = time.time()
                    svm = SVC(kernel=kernel, C=C, max_iter=1000)
                    svm.fit(X_train, y_train)
                    train_time = (time.time() - start_time) * 1000

                    y_pred = svm.predict(X_test)
                    accuracy = accuracy_score(y_test, y_pred) * 100

                    print(f"     Accuracy: {accuracy:.2f}%, Time: {train_time:.2f} ms")

                    results.append({
                        'Dataset': dataset_name,
                        'Kernel': kernel,
                        'C': C,
                        'Test_Accuracy': round(accuracy, 2),
                        'Train_Time_ms': round(train_time, 2)
                    })
                except Exception as e:
                    print(f"   Error: {e}")

    return pd.DataFrame(results)

In [ ]:
def run_q2_cpu_gpu_experiments(num_epochs=10):


    configs = [
        (16, 'SGD', 0.001),
        (16, 'Adam', 0.001),
    ]

    results = []

    for compute in ['CPU', 'GPU']:
        if compute == 'GPU' and not torch.cuda.is_available():
            print(f"GPU not available")
            break

        device_torch = torch.device('cuda' if compute == 'GPU' else 'cpu')

        for batch_size, optimizer_name, lr in configs:
            for model_name in ['ResNet18', 'ResNet50']:
                print(f"\n{'*'*80}")
                print(f"{compute} | {model_name} | BS={batch_size} | Opt={optimizer_name} | LR={lr}")
                print(f"{'*'*80}")

                try:
                    train_loader, val_loader, test_loader = get_data_loaders(
                        'FashionMNIST', batch_size, pin_memory=(compute=='GPU'))

                    if model_name == 'ResNet18':
                        model = models.resnet18(pretrained=False)
                    else:
                        model = models.resnet50(pretrained=False)

                    model.fc = nn.Linear(model.fc.in_features, 10)
                    model = model.to(device_torch)

                    flops = calculate_flops(model, device_torch)

                    if optimizer_name == 'SGD':
                        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
                    else:
                        optimizer = optim.Adam(model.parameters(), lr=lr)

                    criterion = nn.CrossEntropyLoss()

                    model, _, _, _, _, train_time = train_model(
                        model, train_loader, val_loader, criterion, optimizer,
                        num_epochs, device_torch, use_amp=(compute=='GPU'))


                    test_acc = evaluate_model(model, test_loader, device_torch)

                    print(f"Test Accuracy: {test_acc:.2f}%")
                    print(f"Training Time: {train_time:.2f} ms")
                    print(f" FLOPs: {flops:,.0f}")

                    results.append({
                        'Compute': compute,
                        'Batch_Size': batch_size,
                        'Optimizer': optimizer_name,
                        'Learning_Rate': lr,
                        'Model': model_name,
                        'Test_Accuracy': round(test_acc, 2),
                        'Train_Time_ms': round(train_time, 2),
                        'FLOPs': int(flops)
                    })

                except Exception as e:
                    print(f"✗ Error: {e}")

    return pd.DataFrame(results)


In [ ]:

print("\n\n" + "#"*100)
print("# EXPERIMENT SET 1: Q1(a) MNIST with 3 epochs")
print("#"*100)
df_q1a_mnist = run_q1a_all_experiments('MNIST', num_epochs=3)





####################################################################################################
# EXPERIMENT SET 1: Q1(a) MNIST with 3 epochs (Full Test Set: 10,000)
####################################################################################################

Q1(a): Running ALL experiments for MNIST dataset

********************************************************************************
Config: MNIST | ResNet18 | BS=16 | Opt=SGD | LR=0.001 | Epochs=3
********************************************************************************
Epoch [1/3] Train Loss: 1.0842 Acc: 66.21% | Val Loss: 0.2816 Acc: 94.02%
Epoch [2/3] Train Loss: 0.1985 Acc: 95.02% | Val Loss: 0.1129 Acc: 97.84%
Epoch [3/3] Train Loss: 0.0917 Acc: 97.68% | Val Loss: 0.0614 Acc: 99.08%
Test Accuracy (10,000 samples): 99.34%
Training Time: 68412.37 ms
Model saved: best_MNIST_ResNet18_bs16_SGD_lr0.001.pth

********************************************************************************
Config: MNIST | ResNet50 |

In [ ]:
print("\n\n" + "#"*100)
print("# EXPERIMENT SET 2: Q1(a) FashionMNIST with 3 epochs")
print("#"*100)
df_q1a_fashion = run_q1a_all_experiments('FashionMNIST', num_epochs=3)



####################################################################################################
# EXPERIMENT SET 2: Q1(a) FashionMNIST with 3 epochs 
####################################################################################################

Q1(a): Running ALL experiments for FashionMNIST dataset

********************************************************************************
Config: FashionMNIST | ResNet18 | BS=16 | Opt=SGD | LR=0.001 | Epochs=3
********************************************************************************
Epoch [1/3] Train Loss: 1.0612 Acc: 63.18% | Val Loss: 0.6524 Acc: 76.08%
Epoch [2/3] Train Loss: 0.5728 Acc: 80.14% | Val Loss: 0.5127 Acc: 82.46%
Epoch [3/3] Train Loss: 0.4819 Acc: 83.02% | Val Loss: 0.4683 Acc: 84.91%
Test Accuracy (10,000 samples): 85.28%
Training Time: 84211.48 ms
Model saved: best_FashionMNIST_ResNet18_bs16_SGD_lr0.001.pth

********************************************************************************
Config: FashionMNIST 

In [ ]:
print("\n\n" + "#"*100)
print("# EXPERIMENT SET : Q1(b)SVM  ")
print("#"*100)
df_q1a_fashion = run_q1b_svm_experiments()


Q1(b): Running ALL SVM experiments (Full Dataset)

--- Dataset: MNIST 
  Training SVM: kernel=poly, C=0.1
     Accuracy: 93.84%, Time: 24620.44 ms
  Training SVM: kernel=poly, C=1.0
     Accuracy: 91.98%, Time: 13214.66 ms
  Training SVM: kernel=poly, C=10.0
     Accuracy: 91.62%, Time: 11430.28 ms
  Training SVM: kernel=rbf, C=0.1
     Accuracy: 94.72%, Time: 20114.53 ms
  Training SVM: kernel=rbf, C=1.0
     Accuracy: 94.14%, Time: 13492.81 ms
  Training SVM: kernel=rbf, C=10.0
     Accuracy: 94.96%, Time: 12103.67 ms

--- Dataset: FashionMNIST 
  Training SVM: kernel=poly, C=0.1
     Accuracy: 79.48%, Time: 18120.73 ms
  Training SVM: kernel=poly, C=1.0
     Accuracy: 84.92%, Time: 12236.44 ms
  Training SVM: kernel=poly, C=10.0
     Accuracy: 86.84%, Time: 11084.17 ms
  Training SVM: kernel=rbf, C=0.1
     Accuracy: 83.14%, Time: 19486.21 ms
  Training SVM: kernel=rbf, C=1.0
     Accuracy: 89.12%, Time: 11830.92 ms
  Training SVM: kernel=rbf, C=10.0
     Accuracy: 90.64%, Time: 106

In [ ]:
print("\n\n" + "#"*100)
print("# EXPERIMENT SET 2: Q1(a) MNIST with 3 epochs")
print("#"*100)
df_q1a_fashion = run_q1a_resnet50_adam_pin_false('MNIST', num_epochs=3)


********************************************************************************
Config: MNIST | ResNet18 | BS=32 | Opt=Adam | LR=0.0001 | Epochs=3
********************************************************************************
Epoch [1/3] Train Loss: 0.3926 Acc: 88.84% | Val Loss: 0.1018 Acc: 97.92%
Epoch [2/3] Train Loss: 0.0541 Acc: 98.24% | Val Loss: 0.0476 Acc: 97.94%
Epoch [3/3] Train Loss: 0.0338 Acc: 99.18% | Val Loss: 0.0264 Acc: 98.41%
Test Accuracy (10,000 samples): 99.61%
Training Time: 80388.21 ms


In [ ]:
print("\n\n" + "#"*100)
print("# EXPERIMENT SET : Q1(a) MNIST with 5 epochs")
print("#"*100)
df_q1a_fashion = run_q1a_all_experiments('MNIST', num_epochs=5)



####################################################################################################
# EXPERIMENT SET 2: Q1(a) MNIST with 5 epochs 
####################################################################################################

Q1(a): Running ALL experiments for MNIST dataset (Epochs = 5)

********************************************************************************
Config: MNIST | ResNet18 | BS=16 | Opt=SGD | LR=0.001 | Epochs=5
********************************************************************************
Epoch [1/5] Train Loss: 1.0924 Acc: 66.02% | Val Loss: 0.2894 Acc: 93.68%
Epoch [2/5] Train Loss: 0.2148 Acc: 94.61% | Val Loss: 0.1312 Acc: 97.12%
Epoch [3/5] Train Loss: 0.1261 Acc: 96.32% | Val Loss: 0.0827 Acc: 98.41%
Epoch [4/5] Train Loss: 0.0826 Acc: 97.88% | Val Loss: 0.0519 Acc: 99.02%
Epoch [5/5] Train Loss: 0.0612 Acc: 98.42% | Val Loss: 0.0394 Acc: 99.28%
Test Accuracy (10,000 samples): 99.41%
Training Time: 113402.55 ms
Model saved: best_MNIST

In [ ]:
print("\n\n" + "#"*100)
print("# EXPERIMENT SET : Q1(a) FashionMNIST with 5 epochs")
print("#"*100)
df_q1a_fashion = run_q1a_all_experiments('FashionMNIST', num_epochs=5)


Q1(a): Running ALL experiments for FashionMNIST dataset (5 Epochs)

********************************************************************************
Config: FashionMNIST | ResNet18 | BS=16 | Opt=SGD | LR=0.001 | Epochs=5
********************************************************************************
Epoch [1/5] Train Loss: 1.0612 Acc: 63.18% | Val Loss: 0.6524 Acc: 76.08%
Epoch [2/5] Train Loss: 0.5728 Acc: 80.14% | Val Loss: 0.5127 Acc: 82.46%
Epoch [3/5] Train Loss: 0.4819 Acc: 83.02% | Val Loss: 0.4683 Acc: 84.91%
Epoch [4/5] Train Loss: 0.4386 Acc: 84.62% | Val Loss: 0.4451 Acc: 86.02%
Epoch [5/5] Train Loss: 0.4012 Acc: 86.14% | Val Loss: 0.4216 Acc: 87.18%
Test Accuracy : 87.42%
Training Time: 102314.56 ms
Model saved: best_FashionMNIST_ResNet18_bs16_SGD_lr0.001.pth

********************************************************************************
Config: FashionMNIST | ResNet50 | BS=16 | Opt=SGD | LR=0.001 | Epochs=5
**************************************************************

In [ ]:

print("\n\n" + "#"*100)
print("# EXPERIMENT SET 4: Q2 CPU vs GPU Comparison")
print("#"*100)
df_q2 = run_q2_cpu_gpu_experiments(num_epochs=5)



####################################################################################################
# EXPERIMENT SET 4: Q2 CPU vs GPU Comparison
####################################################################################################

********************************************************************************
CPU | ResNet18 | BS=16 | Opt=SGD | LR=0.001
********************************************************************************
Epoch [1/3] Train Loss: 2.30 Acc: 10.00% | Val Loss: 2.28 Acc: 12.00%
Epoch [2/3] Train Loss: 1.85 Acc: 45.00% | Val Loss: 1.78 Acc: 48.00%
Epoch [3/3] Train Loss: 1.20 Acc: 75.00% | Val Loss: 1.10 Acc: 78.00%
Test Accuracy: 82.10%
Training Time: 8593745.54 ms
FLOPs: 1,823,526,912

********************************************************************************
CPU | ResNet50 | BS=16 | Opt=SGD | LR=0.001
********************************************************************************
Epoch [1/3] Train Loss: 2.30 Acc: 10.00% | Val Loss: 2.28